# CS781 Phase 3 — Rhetorical Role Prediction (Model 3: encoder–decoder LM, `ours_encdec*`)
Starting point: our own Phase-1 encoder–decoder legal LM (no external weights; 6-layer bidirectional encoder + 6-layer cross-attention decoder). Only the shared embeddings and the **bidirectional encoder** are used as the backbone for sentence classification; the decoder is dropped. Each sentence gets one of 13 rhetorical roles; CL and IT are trained jointly and evaluated separately. Experiments in this notebook:
1. **LoRMA+** (frozen encoder, multiplicative low-rank adaptation), **context-aware** input (windows of consecutive sentences). Also the distillation teacher.
2. **Knowledge distillation**: half-depth encoder student (initialised from Phase-1 layers, fully fine-tuned) learns from the LoRMA+ teacher's sentence-level logits
3. **Prefix tuning** (frozen encoder, learned key/value prefixes), context-aware
4. **Houlsby bottleneck adapters** (frozen encoder), context-aware
5. **LoRMA+ with sentence-only input** (ablation: does document context help?)

Sentence representations are mean-pooled encoder token states followed by a linear classifier. Each experiment writes a `prediction.json` (one label per sentence) and a zip.

## 1. Setup and imports

In [ ]:
import os
import sys
import json
import glob
import time
import math
import pickle
import random
import zipfile
import subprocess

import numpy as np


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args])


try:
    import regex
except ImportError:
    pip_install("regex")
    import regex

gpu_check = "import jax, sys; sys.exit(jax.devices()[0].platform not in ('gpu', 'cuda'))"
if subprocess.run([sys.executable, "-c", gpu_check]).returncode != 0:
    pip_install("-U", "jax[cuda12]")

import jax
import jax.numpy as jnp
from jax import lax
from huggingface_hub import hf_hub_download, login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF"))
except Exception:
    pass

T0 = time.time()
random.seed(0)
np.random.seed(0)
tmap = jax.tree_util.tree_map
print("devices:", jax.devices())

## 2. Hyperparameters and time budget

In [ ]:
CONTEXT_TOKENS = 512
SENT_TOKENS = 128
MAX_SENTS = 48
TRAIN_TOKENS = 4096
PRED_TOKENS = 8192
RANK = 16
PEFT_ALPHA = 16
PEFT_SCALE = PEFT_ALPHA / RANK
ADAPTER_DIM = 64
PREFIX_LEN = 32
PREFIX_LR_MULT = 4.0
KD_TEMP = 2.0
KD_ALPHA = 0.5
MAX_EPOCHS = 30
PATIENCE = 6
WARMUP_STEPS = 100
BIG = 1 << 30
EOS = 3
MAX_SECONDS = 7.5 * 3600
LORMA_MINUTES = 70
KD_MINUTES = 60
PREFIX_MINUTES = 60
ADAPTER_MINUTES = 60
SENTENCE_MINUTES = 40


def budget(minutes):
    return min(minutes * 60, MAX_SECONDS - (time.time() - T0) - 2400)

## 3. Load the Phase-1 checkpoint and tokenizer

In [ ]:
CK_DIR = None  # optionally set to the folder holding the encoder-decoder checkpoint

# The Phase-1 notebook writes final_params.pkl = dict(params, cfg, variant) for the BEST "ours_*" variant of its
# architecture ablation only, so the file is an encoder-decoder only if ours_encdec(_copy) won (or BEST was forced).
# It also writes final_ckpt.pkl = dict(params, m, v, step, ...) for the run it trained, without cfg; the config of the
# encoder-decoder variants is therefore repeated here for that case.
ENCDEC_CFG = dict(norm="rms", mlp="swiglu", pos="rope", kvh=4, qknorm=True, hybrid=False, gate=False, ff=2048,
                  encdec=True, copy=False, le=6, ld=6)


def read_checkpoint(path):
    d = pickle.load(open(path, "rb"))
    params, c = d["params"], d.get("cfg")
    if c is None and "enc" in params and "dec" in params:
        c = dict(ENCDEC_CFG, copy="wcq" in params)
    return params, c, d.get("variant", "(not stored)")


if CK_DIR:
    candidates = [p for p in (f"{CK_DIR}/final_params.pkl", f"{CK_DIR}/final_ckpt.pkl") if os.path.exists(p)]
else:
    candidates = (sorted(glob.glob("/kaggle/input/**/final_params.pkl", recursive=True))
                  + sorted(glob.glob("/kaggle/input/**/final_ckpt.pkl", recursive=True)))
print("checkpoint files found:", candidates)

params = cfg = None
for path in candidates:
    p_, c_, variant = read_checkpoint(path)
    ok = bool(c_) and bool(c_.get("encdec")) and bool(c_.get("qknorm")) and "enc" in p_ and "ne" in p_
    print(f"  {path}\n    variant {variant} | encdec={(c_ or {}).get('encdec')} | top-level params {list(p_)[:10]} -> {'USE' if ok else 'skip'}")
    if ok:
        params, cfg, CK_DIR = p_, c_, os.path.dirname(path)
        break
    del p_
assert params is not None, (
    "none of the files above is an encoder-decoder checkpoint. The Phase-1 notebook only saves the best ours_* variant "
    "of its ablation as final_params.pkl; attach the output of the run that trained ours_encdec (or set BEST = 'ours_encdec' "
    "before its final-training cell), or set CK_DIR to the folder holding it")

# only the embeddings and the bidirectional encoder are needed; the decoder, cross-attention and copy head are dropped
P0 = tmap(jnp.asarray, {k: params[k] for k in ("emb", "enc", "ne")})
n_dropped = sum(int(np.prod(x.shape)) for k in params if k not in P0 for x in jax.tree_util.tree_leaves(params[k]))
del params
print("using", CK_DIR, "| config:", cfg, f"| dropped decoder-side parameters: {n_dropped / 1e6:.1f}M")

tk_path = f"{CK_DIR}/bpe_legal.json"
if not os.path.exists(tk_path):
    tk_path = sorted(glob.glob("/kaggle/input/**/bpe_legal.json", recursive=True))[0]
tk = json.load(open(tk_path))

V, D = P0["emb"].shape
HD = 64
H = P0["enc"]["wq"].shape[2] // HD
KV = cfg["kvh"]
R = H // KV
N_ENC = P0["enc"]["wq"].shape[0]
n_enc_params = sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(P0["enc"])) + V * D
print(f"vocab {V}, dim {D}, encoder layers {N_ENC}, heads {H}, kv heads {KV}, encoder backbone (with embeddings) {n_enc_params / 1e6:.1f}M")
assert V >= len(tk["itos"]), f"embedding has {V} rows but the tokenizer has {len(tk['itos'])} symbols: wrong bpe_legal.json for this checkpoint"

## 4. Sentence tokenizer (byte-level BPE from Phase 1)

In [ ]:
PAT = regex.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

byte_list = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
char_list = byte_list[:]
extra = 0
for b in range(256):
    if b not in byte_list:
        byte_list.append(b)
        char_list.append(256 + extra)
        extra += 1
B2U = {b: chr(c) for b, c in zip(byte_list, char_list)}

stoi = {s: i for i, s in enumerate(tk["itos"])}
rank = {tuple(m): i for i, m in enumerate(tk["merges"])}
bpe_cache = {}


def bpe(word):
    if word not in bpe_cache:
        syms = [B2U[b] for b in word.encode()]
        while len(syms) > 1:
            best_rank, best_i = min((rank.get(p, BIG), i) for i, p in enumerate(zip(syms, syms[1:])))
            if best_rank == BIG:
                break
            syms[best_i:best_i + 2] = [syms[best_i] + syms[best_i + 1]]
        bpe_cache[word] = [stoi[s] for s in syms]
    return bpe_cache[word]


def encode_sentence(sentence):
    ids = []
    for w in PAT.findall(" " + " ".join(str(sentence).split())):
        ids.extend(bpe(w))
    return ids[:SENT_TOKENS - 1] + [EOS]

## 5. Load Phase-3 data (CL and IT train / dev / test) and the label vocabulary

In [ ]:
def hf_file(name):
    return hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{name}", repo_type="dataset", local_dir="/tmp/hf")


def load_docs(name):
    data = json.load(open(hf_file(name)))
    if isinstance(data, dict):
        data = [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    return data


def parse_labels(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if all(str(k).isdigit() for k in raw):
        return [raw[k] for k in sorted(raw, key=int)]
    if all(isinstance(v, int) for v in raw.values()):
        return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    return list(raw)


LABELS = parse_labels(json.load(open(hf_file("Train/label_vocab.json"))))
K = len(LABELS)
LAB_ID = {l: i for i, l in enumerate(LABELS)}
print(K, "labels:", LABELS)

DOCS = {}
for dom in ("CL", "IT"):
    DOCS[dom] = {}
    for split in ("train", "dev", "test"):
        folder = "Test" if split == "test" else "Train"
        raw = load_docs(f"{folder}/{dom}_{split}.json")
        DOCS[dom][split] = [
            {
                "id": str(d["id"]),
                "sent_ids": [encode_sentence(s) for s in d["text"]],
                "y": np.array([LAB_ID[l] for l in d["labels"]], np.int32) if "labels" in d else None,
            }
            for d in raw
        ]
        n_sent = sum(len(d["sent_ids"]) for d in DOCS[dom][split])
        n_tok = sum(len(s) for d in DOCS[dom][split] for s in d["sent_ids"])
        print(f"{dom} {split}: {len(DOCS[dom][split])} docs, {n_sent} sentences, {n_tok / max(n_sent, 1):.1f} tokens/sentence")

for dom in ("CL", "IT"):
    y = np.concatenate([d["y"] for d in DOCS[dom]["train"]])
    print(dom, "train label counts:", dict(zip(LABELS, np.bincount(y, minlength=K).tolist())))

ALL_TRAIN = DOCS["CL"]["train"] + DOCS["IT"]["train"]
counts = np.bincount(np.concatenate([d["y"] for d in ALL_TRAIN]), minlength=K).astype(np.float64)
CLASS_W = np.where(counts > 0, np.clip((np.median(counts[counts > 0]) / np.maximum(counts, 1)) ** 0.5, 0.25, 4.0), 1.0).astype(np.float32)
print("class weights:", dict(zip(LABELS, CLASS_W.round(2).tolist())))

## 6. Sentence-only and context-aware windows

In [ ]:
def make_windows(sent_ids, mode):
    n = len(sent_ids)
    if mode == "sentence":
        return [(i, i + 1) for i in range(n)]
    out, s = [], 0
    while s < n:
        e, tok = s, 0
        while e < n and e - s < MAX_SENTS and tok + len(sent_ids[e]) <= CONTEXT_TOKENS:
            tok += len(sent_ids[e])
            e += 1
        e = max(e, s + 1)
        out.append((s, e))
        if e >= n:
            break
        s += max(1, (e - s) // 2)
    return out


def build_set(docs, mode):
    T = SENT_TOKENS if mode == "sentence" else CONTEXT_TOKENS
    S = 1 if mode == "sentence" else MAX_SENTS
    ids, sidx, ys, meta = [], [], [], []
    for di, d in enumerate(docs):
        for s, e in make_windows(d["sent_ids"], mode):
            row = np.zeros(T, np.int32)
            si = np.full(T, -1, np.int32)
            y = np.full(S, -1, np.int32)
            pos = 0
            for j in range(s, e):
                seg = d["sent_ids"][j]
                row[pos:pos + len(seg)] = seg
                si[pos:pos + len(seg)] = j - s
                pos += len(seg)
                if d["y"] is not None:
                    y[j - s] = d["y"][j]
            ids.append(row)
            sidx.append(si)
            ys.append(y)
            meta.append((di, s, e))
    return {"ids": np.stack(ids), "sidx": np.stack(sidx), "y": np.stack(ys), "meta": meta}


SETS = {}


def get_sets(mode):
    if mode not in SETS:
        SETS[mode] = {
            "train": build_set(ALL_TRAIN, mode),
            "dev": {dom: build_set(DOCS[dom]["dev"], mode) for dom in ("CL", "IT")},
            "test": {dom: build_set(DOCS[dom]["test"], mode) for dom in ("CL", "IT")},
        }
        print(mode, "windows: train", len(SETS[mode]["train"]["ids"]), "| dev",
              {d: len(v["ids"]) for d, v in SETS[mode]["dev"].items()})
    return SETS[mode]


for m in ("context", "sentence"):
    get_sets(m)

## 7. Model: Phase-1 bidirectional encoder (frozen or adapted) + sentence pooling + linear classifier

In [ ]:
inv_freq = 1 / (10000 ** (np.arange(0, HD, 2) / HD))
angles = np.arange(1280)[:, None] * inv_freq
COS = jnp.asarray(np.cos(angles), jnp.float32)
SIN = jnp.asarray(np.sin(angles), jnp.float32)
CLASS_W_J = jnp.asarray(CLASS_W)
LOGIT_SENTS = {SENT_TOKENS: 1, CONTEXT_TOKENS: MAX_SENTS}
ADAPTED = ("wq", "wk", "wv", "wo", "w1", "w2", "w3")
HEAD_KEYS = ("nh", "cls", "cb")
PREFIX_LEAVES = ("pk", "pv")
PEFT_SUFFIX = ("_ma", "_mb", "_dn", "_up")


def rmsn(x, w):
    return x * lax.rsqrt((x * x).mean(-1, keepdims=True) + 1e-6) * w


def rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2], x[..., HD // 2:]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)


def lin(p, name, x):
    y = x @ p[name]
    if name + "_mb" in p:
        # LoRMA+: multiplicative low-rank update acting on the output of the frozen weight
        y = y + PEFT_SCALE * (y @ p[name + "_ma"]) @ p[name + "_mb"]
    return y


def adapt(p, name, y):
    # Houlsby bottleneck adapter: residual down-project -> GELU -> up-project
    if name + "_dn" in p:
        y = y + jax.nn.gelu(y @ p[name + "_dn"]) @ p[name + "_up"]
    return y


def encoder_layer(p, x, cos, sin, valid):
    Bsz, T, _ = x.shape
    h = rmsn(x, p["n1"])
    q = rope(rmsn(lin(p, "wq", h).reshape(Bsz, T, H, HD), p["qn"]), cos, sin)
    k = rope(rmsn(lin(p, "wk", h).reshape(Bsz, T, KV, HD), p["kn"]), cos, sin)
    v = lin(p, "wv", h).reshape(Bsz, T, KV, HD)

    key_valid = valid
    if "pk" in p:
        # Prefix tuning: learned key/value vectors prepended to every layer's attention
        n_prefix = p["pk"].shape[0]
        pk = jnp.broadcast_to(rmsn(p["pk"], p["kn"])[None], (Bsz, n_prefix, KV, HD))
        pv = jnp.broadcast_to(p["pv"][None], (Bsz, n_prefix, KV, HD))
        k = jnp.concatenate([pk, k], 1)
        v = jnp.concatenate([pv, v], 1)
        key_valid = jnp.concatenate([jnp.ones((Bsz, n_prefix), bool), valid], 1)

    scores = jnp.einsum("btgrd,bsgd->bgrts", q.reshape(Bsz, T, KV, R, HD), k) * HD ** -0.5
    attn = jax.nn.softmax(jnp.where(key_valid[:, None, None, None, :], scores, -1e30), -1)
    out = lin(p, "wo", jnp.einsum("bgrts,bsgd->btgrd", attn, v).reshape(Bsz, T, H * HD))
    x = x + adapt(p, "a1", out)

    h = rmsn(x, p["n2"])
    ff = lin(p, "w2", jax.nn.silu(lin(p, "w1", h)) * lin(p, "w3", h))
    return x + adapt(p, "a2", ff)


def logits_fn(P, ids, sidx):
    T = ids.shape[1]
    n_sent = LOGIT_SENTS[T]
    cos, sin = COS[:T][None, :, None, :], SIN[:T][None, :, None, :]
    valid = sidx >= 0

    x, _ = lax.scan(lambda x, lp: (encoder_layer(lp, x, cos, sin, valid), None), P["emb"][ids], P["enc"])
    x = rmsn(x, P["ne"])

    onehot = jax.nn.one_hot(sidx, n_sent)
    pooled = jnp.einsum("bts,btd->bsd", onehot, x) / jnp.maximum(onehot.sum(1), 1.0)[..., None]
    return rmsn(pooled, P["nh"]) @ P["cls"] + P["cb"]


def build_model(enc, adapter, key):
    enc = dict(enc)
    n_layers = enc["wq"].shape[0]
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    if adapter == "lorma":
        for i, name in enumerate(ADAPTED):
            _, d_in, d_out = enc[name].shape
            enc[name + "_ma"] = jax.random.normal(jax.random.fold_in(k1, i), (n_layers, d_out, RANK)) * 0.02
            enc[name + "_mb"] = jnp.zeros((n_layers, RANK, d_out))
    elif adapter == "adapter":
        for i, name in enumerate(("a1", "a2")):
            enc[name + "_dn"] = jax.random.normal(jax.random.fold_in(k2, i), (n_layers, D, ADAPTER_DIM)) * 0.02
            enc[name + "_up"] = jnp.zeros((n_layers, ADAPTER_DIM, D))
    elif adapter == "prefix":
        enc["pk"] = jax.random.normal(k3, (n_layers, PREFIX_LEN, KV, HD)) * 0.02
        enc["pv"] = jax.random.normal(k4, (n_layers, PREFIX_LEN, KV, HD)) * 0.02
    return {
        "emb": P0["emb"],
        "enc": enc,
        "ne": P0["ne"],
        "nh": jnp.ones(D),
        "cls": jax.random.normal(k5, (D, K)) * 0.02,
        "cb": jnp.zeros(K),
    }


def trainable_mask(P, method):
    if method == "full":
        return jax.tree_util.tree_map_with_path(lambda path, x: True, P)

    def is_trainable(path, x):
        top, leaf = path[0].key, str(path[-1].key)
        return top in HEAD_KEYS or (top == "enc" and (leaf in PREFIX_LEAVES or leaf.endswith(PEFT_SUFFIX)))
    return jax.tree_util.tree_map_with_path(is_trainable, P)


def lr_scale(path, head_mult):
    top, leaf = path[0].key, str(path[-1].key)
    if top in HEAD_KEYS:
        return head_mult
    if leaf in PREFIX_LEAVES:
        return PREFIX_LR_MULT
    return 1.0


def count_params(P, mask=None):
    leaves = jax.tree_util.tree_leaves(P)
    if mask is None:
        return sum(int(np.prod(x.shape)) for x in leaves)
    return sum(int(np.prod(x.shape)) for x, tr in zip(leaves, jax.tree_util.tree_leaves(mask)) if tr)


logp_fn = jax.jit(lambda P, ids, sidx: jax.nn.log_softmax(logits_fn(P, ids, sidx)))
logits_jit = jax.jit(logits_fn)

## 8. Prediction (one label per sentence) and Macro-F1 / Precision / Recall

In [ ]:
def predict_scores(P, data, docs):
    T = data["ids"].shape[1]
    pb = PRED_TOKENS // T
    acc = [np.zeros((len(d["sent_ids"]), K), np.float32) for d in docs]
    cnt = [np.zeros(len(d["sent_ids"]), np.float32) for d in docs]
    for i in range(0, len(data["ids"]), pb):
        ids, sidx = data["ids"][i:i + pb], data["sidx"][i:i + pb]
        pad = pb - len(ids)
        if pad:
            ids = np.concatenate([ids, np.zeros((pad, T), np.int32)])
            sidx = np.concatenate([sidx, np.full((pad, T), -1, np.int32)])
        lp = np.asarray(logp_fn(P, ids, sidx))
        for j, (di, s, e) in enumerate(data["meta"][i:i + pb]):
            acc[di][s:e] += lp[j, :e - s]
            cnt[di][s:e] += 1
    return [a / c[:, None] for a, c in zip(acc, cnt)]


def macro_scores(gold, pred):
    present = sorted(set(gold.tolist()) | set(pred.tolist()))
    f1_all = np.zeros(K)
    precs, recs, f1s = [], [], []
    for k in present:
        tp = int(((gold == k) & (pred == k)).sum())
        fp = int(((gold != k) & (pred == k)).sum())
        fn = int(((gold == k) & (pred != k)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        precs.append(pr)
        recs.append(rc)
        f1s.append(f1)
        f1_all[k] = f1
    return float(np.mean(f1s)), float(np.mean(precs)), float(np.mean(recs)), f1_all


def dev_predictions(P, mode):
    sets = get_sets(mode)
    out = {}
    for dom in ("CL", "IT"):
        scores = predict_scores(P, sets["dev"][dom], DOCS[dom]["dev"])
        pred = np.concatenate([s.argmax(1) for s in scores])
        gold = np.concatenate([d["y"] for d in DOCS[dom]["dev"]])
        out[dom] = (gold, pred)
    return out


def evaluate(P, mode):
    out = dev_predictions(P, mode)
    res = {dom: macro_scores(*out[dom]) for dom in out}
    pooled = macro_scores(np.concatenate([out[d][0] for d in out]), np.concatenate([out[d][1] for d in out]))
    return (res["CL"][0] + res["IT"][0]) / 2, res, pooled


def save_submission(name, P, mode):
    t_start = time.time()
    sets = get_sets(mode)
    result = {}
    for dom in ("CL", "IT"):
        scores = predict_scores(P, sets["test"][dom], DOCS[dom]["test"])
        for d, s in zip(DOCS[dom]["test"], scores):
            assert len(s) == len(d["sent_ids"])
            if d["id"] in result:
                print("warning: duplicate document id across domains:", d["id"])
            result[d["id"]] = [LABELS[i] for i in s.argmax(1)]
    elapsed = time.time() - t_start
    out_dir = f"/kaggle/working/{name}"
    os.makedirs(out_dir, exist_ok=True)
    json.dump(result, open(f"{out_dir}/prediction.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{out_dir}/prediction.json", "prediction.json")
    print(f"{name}: {len(result)} documents, {sum(len(v) for v in result.values())} sentences, inference {elapsed:.0f}s")
    return elapsed

## 9. Training loop (class-weighted CE, optional distillation loss, frozen weights via stop-gradient, early stopping)

In [ ]:
def make_step(mask, lr_mult):
    def loss_fn(P, ids, sidx, y, teacher_logits, alpha):
        Pe = tmap(lambda p, tr: p if tr else lax.stop_gradient(p), P, mask)
        logits = logits_fn(Pe, ids, sidx)
        logp = jax.nn.log_softmax(logits)
        keep = y >= 0
        yy = jnp.maximum(y, 0)
        w = CLASS_W_J[yy] * keep
        ce = (-jnp.take_along_axis(logp, yy[..., None], -1)[..., 0] * w).sum() / w.sum()

        t_logp = jax.nn.log_softmax(teacher_logits / KD_TEMP)
        s_logp = jax.nn.log_softmax(logits / KD_TEMP)
        kd = ((jnp.exp(t_logp) * (t_logp - s_logp)).sum(-1) * keep).sum() / jnp.maximum(keep.sum(), 1) * KD_TEMP ** 2
        return (1 - alpha) * ce + alpha * kd

    @jax.jit
    def step(P, m, v, t, lr, ids, sidx, y, teacher_logits, alpha):
        loss, g = jax.value_and_grad(loss_fn)(P, ids, sidx, y, teacher_logits, alpha)
        gnorm = jnp.sqrt(sum((x ** 2).sum() for x in jax.tree_util.tree_leaves(g)))
        g = tmap(lambda x: x * jnp.minimum(1.0, 1.0 / (gnorm + 1e-6)), g)
        m = tmap(lambda a, b: 0.9 * a + 0.1 * b, m, g)
        v = tmap(lambda a, b: 0.999 * a + 0.001 * b * b, v, g)

        def update(p, a, b, mult):
            adam = (a / (1 - 0.9 ** t)) / (jnp.sqrt(b / (1 - 0.999 ** t)) + 1e-8)
            return p - lr * mult * (adam + 0.01 * p)

        return tmap(update, P, m, v, lr_mult), m, v, loss

    return step


def train(name, P, mode, method, minutes, peak_lr, head_mult, teacher_logits=None, alpha=0.0):
    mask = trainable_mask(P, method)
    lr_mult = jax.tree_util.tree_map_with_path(
        lambda path, x, tr: float(tr) * lr_scale(path, head_mult), P, mask)
    step = make_step(mask, lr_mult)
    m = tmap(jnp.zeros_like, P)
    v = tmap(jnp.zeros_like, P)
    data = get_sets(mode)["train"]
    T = data["ids"].shape[1]
    batch = TRAIN_TOKENS // T
    S = data["y"].shape[1]
    zeros = jnp.zeros((batch, S, K))
    steps_per_epoch = max(len(data["ids"]) // batch, 1)
    seconds = budget(minutes)
    print(f"[{name}] mode {mode} | trainable {count_params(P, mask) / 1e6:.2f}M of {count_params(P) / 1e6:.1f}M | {steps_per_epoch} steps/epoch")

    t, total, done, bad = 0, None, False, 0
    best = (-1, None, None)
    t_start = time.time()

    while not done:
        perm = np.random.permutation(len(data["ids"]))
        for i in range(steps_per_epoch):
            ix = perm[i * batch:(i + 1) * batch]
            t += 1
            lr = peak_lr * min(1, t / WARMUP_STEPS)
            if total:
                lr *= 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, t / total)))
            tl = jnp.asarray(teacher_logits[ix], jnp.float32) if teacher_logits is not None else zeros
            P, m, v, loss = step(P, m, v, jnp.float32(t), lr, data["ids"][ix], data["sidx"][ix], data["y"][ix], tl, jnp.float32(alpha))
            if t == 5:
                loss.block_until_ready()
                t_bench = time.time()
            if t == 25:
                loss.block_until_ready()
                sec_per_step = (time.time() - t_bench) / 20
                total = min(MAX_EPOCHS * steps_per_epoch, int(0.9 * seconds / sec_per_step))
                print(f"[{name}] {sec_per_step:.2f}s/step -> {total} steps ({total / steps_per_epoch:.1f} epochs)")
            if (total and t >= total) or time.time() - T0 > MAX_SECONDS:
                done = True
                break

        mean_f1, res, pooled = evaluate(P, mode)
        print(f"[{name}] step {t} loss {float(loss):.4f} | dev macro-F1 CL {res['CL'][0]:.4f} IT {res['IT'][0]:.4f} mean {mean_f1:.4f} "
              f"| {(time.time() - t_start) / 60:.1f} min", flush=True)
        if mean_f1 > best[0]:
            best, bad = (mean_f1, P, (res, pooled)), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] early stopping")
                done = True

    return best, time.time() - t_start

## 10. Experiment 1 — LoRMA+ (rank 16), context-aware windows (also the distillation teacher)

In [ ]:
RESULTS = []

model = build_model(P0["enc"], "lorma", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, teacher, (res, pooled)), train_secs = train("lorma_ctx", model, "context", "peft", LORMA_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_encdec_lorma", teacher, "context")
RESULTS.append(("lorma_ctx", n_trainable, count_params(teacher), best_f1, res, pooled, train_secs, infer_secs))
del model

## 11. Experiment 2 — Knowledge distillation (half-depth encoder student, teacher = Experiment 1)

In [ ]:
data = get_sets("context")["train"]
teacher_logits = np.concatenate([
    np.asarray(logits_jit(teacher, data["ids"][i:i + 16], data["sidx"][i:i + 16]))
    for i in range(0, len(data["ids"]), 16)
]).astype(np.float16)
print("teacher logits:", teacher_logits.shape)

student_idx = np.arange(0, N_ENC, 2)
student_enc = tmap(lambda a: a[student_idx], P0["enc"])
student = build_model(student_enc, None, jax.random.PRNGKey(1))
print(f"teacher encoder layers {N_ENC} -> student encoder layers {len(student_idx)}")
n_trainable = count_params(student)
(best_f1, best_P, (res, pooled)), train_secs = train(
    "kd_student", student, "context", "full", KD_MINUTES, 5e-5, 10.0, teacher_logits=teacher_logits, alpha=KD_ALPHA)
infer_secs = save_submission("p3_encdec_kd", best_P, "context")
RESULTS.append(("kd_student", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del best_P, student, teacher_logits

## 12. Experiment 3 — Prefix tuning (32 learned key/value prefix vectors per layer), context-aware windows

In [ ]:
model = build_model(P0["enc"], "prefix", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("prefix_ctx", model, "context", "peft", PREFIX_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_encdec_prefix", best_P, "context")
RESULTS.append(("prefix_ctx", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del model, best_P

## 13. Experiment 4 — Houlsby bottleneck adapters (bottleneck 64), context-aware windows

In [ ]:
model = build_model(P0["enc"], "adapter", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("adapter_ctx", model, "context", "peft", ADAPTER_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_encdec_adapter", best_P, "context")
RESULTS.append(("adapter_ctx", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del model, best_P

## 14. Experiment 5 — LoRMA+ (rank 16), sentence-only input (context ablation)

In [ ]:
model = build_model(P0["enc"], "lorma", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("lorma_sent", model, "sentence", "peft", SENTENCE_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_encdec_lorma_sentence", best_P, "sentence")
RESULTS.append(("lorma_sent", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del model, best_P

## 15. Results summary

In [ ]:
print(f"{'experiment':12s}{'trainable(M)':>13s}{'total(M)':>10s}{'CL F1':>8s}{'CL P':>8s}{'CL R':>8s}{'IT F1':>8s}{'IT P':>8s}{'IT R':>8s}{'mean F1':>9s}{'train(min)':>11s}{'infer(s)':>9s}")
for name, n_tr, n_tot, mean_f1, res, pooled, tr, inf in RESULTS:
    cl, it = res["CL"], res["IT"]
    print(f"{name:12s}{n_tr / 1e6:13.2f}{n_tot / 1e6:10.1f}{cl[0]:8.4f}{cl[1]:8.4f}{cl[2]:8.4f}{it[0]:8.4f}{it[1]:8.4f}{it[2]:8.4f}{mean_f1:9.4f}{tr / 60:11.1f}{inf:9.0f}")

print()
print(f"{'label (pooled F1)':26s}" + "".join(f"{r[0]:>12s}" for r in RESULTS))
for k, label in enumerate(LABELS):
    print(f"{label:26s}" + "".join(f"{r[5][3][k]:12.4f}" for r in RESULTS))

by_name = {r[0]: r for r in RESULTS}
ctx = by_name["lorma_ctx"]
if "lorma_sent" in by_name:
    sent = by_name["lorma_sent"]
    print(f"\ncontext effect (LoRMA+ mean dev macro-F1): context {ctx[3]:.4f} vs sentence-only {sent[3]:.4f} -> {ctx[3] - sent[3]:+.4f}")
for other in ("prefix_ctx", "adapter_ctx"):
    if other in by_name:
        print(f"LoRMA+ vs {other} (context): {ctx[3]:.4f} vs {by_name[other][3]:.4f} -> {ctx[3] - by_name[other][3]:+.4f}")
if "kd_student" in by_name:
    print(f"distillation: teacher {ctx[3]:.4f} ({ctx[2] / 1e6:.1f}M) vs student {by_name['kd_student'][3]:.4f} ({by_name['kd_student'][2] / 1e6:.1f}M)")
print(f"total time {(time.time() - T0) / 3600:.2f} h")

## 16. Error analysis — per-domain F1 and most frequent confusions (LoRMA+ context model)

In [ ]:
dev = dev_predictions(teacher, "context")
print(f"{'label':26s}{'CL F1':>9s}{'IT F1':>9s}{'CL gold':>9s}{'IT gold':>9s}")
for k, label in enumerate(LABELS):
    row = [by_name["lorma_ctx"][4][dom][3][k] for dom in ("CL", "IT")]
    n_gold = [int((dev[dom][0] == k).sum()) for dom in ("CL", "IT")]
    print(f"{label:26s}{row[0]:9.4f}{row[1]:9.4f}{n_gold[0]:9d}{n_gold[1]:9d}")

for dom, (gold, pred) in dev.items():
    cm = np.zeros((K, K), int)
    np.add.at(cm, (gold, pred), 1)
    off = sorted(((cm[i, j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0), reverse=True)
    print(f"\n{dom}: most frequent confusions (gold -> predicted)")
    for c, a, b in off[:10]:
        print(f"  {a:24s} -> {b:24s} {c:5d}")